In [1]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from selenium import webdriver
from time import sleep
import os

import pdfplumber

In [2]:

# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'MC CCAF'
print(f"Running {regulatorName} Web Scraping Tool v.1.0")
now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment

os.chdir(scriptfolder)


tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)





Running MC CCAF Web Scraping Tool v.1.0


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------


chromeOptions = webdriver.ChromeOptions()
prefs = {"download.default_directory" : tempfolder, 
        "plugins.always_open_pdf_externally": True}
chromeOptions.add_experimental_option("prefs", prefs)
driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

Typology={regulatorName+' 1':'List of authorised firms' , regulatorName+' 2': 'List of funds'}
regdict={
    regulatorName+' 1': 'https://ccaf.mc/en/downloads/',
    regulatorName+' 2': 'https://ccaf.mc/en/downloads/',
        }

reg_hook = {
    regulatorName+' 1': 'authorised firms',
    regulatorName+' 2': 'funds open to all investors',
}
processdate = now.strftime('%Y-%m-%d')

In [4]:

# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


In [5]:
# %%

#------------------------------------------------ Main_Function -------------------------------
from bs4 import BeautifulSoup

for reg in regdict:
    print(f"Working with {reg}")
    driver.get(regdict[reg])
    sleep(3)
    soup = BeautifulSoup(driver.page_source, "html.parser")
    download_lists = soup.find_all('div',class_='downloads-list')
    for list in download_lists:
        if reg_hook[reg] in list.text:
            driver.get(list.find('a')['href'])
    sleep(3)
    pdf_file = os.listdir(tempfolder)[0]
    filePath = os.path.join(tempfolder, pdf_file)

    tables = []
    with pdfplumber.open(filePath) as pdf:
        for page in pdf.pages:
            table = page.extract_table()
            tables.append(table)
    if os.path.exists(tempfolder):
        for rem in os.listdir(tempfolder):
            os.remove(os.path.join(tempfolder, rem))
    else:
        os.mkdir(tempfolder)
    if reg == regulatorName+' 1':
        for table in tables:
            for tab in table:
                if tab[0] and 'ENTITE AGREEE' not in tab[0]:
                    name_ = tab[0]
                    date_ = tab[-1]
                    NUMERO_AGREMENT_INITIAL = tab[-2]
                    #print(name_, date_)
                    sqldict['Name'].append(name_)
                    sqldict['RegulationDate'].append(date_ if date_ and len(date_) > 2 else '')
                    sqldict['InternalID_1'].append(NUMERO_AGREMENT_INITIAL)
                    sqldict['InternalID_1_type'].append('NUMERO D’AGREMENT INITIAL')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)
    elif reg == regulatorName+' 2':
        for table in tables:
            for tab in table:
                if tab[0] and 'DENOMINATION' not in tab[0]:
                    name_ = tab[0].replace('\n',' ')
                    isin_code = tab[1]
                    management_company = tab[2]
                    date_ = tab[-1]
                    NUMERO_AGREMENT_INITIAL = tab[-2]
                    sqldict['Name'].append(name_)
                    sqldict['RegulationDate'].append(date_ if date_ and len(date_) > 2 else '')
                    sqldict['InternalID_1'].append(NUMERO_AGREMENT_INITIAL)
                    sqldict['InternalID_1_type'].append('NUMERO D’AGREMENT INITIAL')
                    sqldict['InternalID_2'].append(isin_code)
                    sqldict['InternalID_2_type'].append('CODE ISIN')
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['RegCtry'].append(reg.split(' ')[0])
                    sqldict['RegCode'].append(reg.split(' ')[1])
                    sqldict['ListCode'].append(reg.split(' ')[-1])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append(Typology[reg])
                    sqldict = bourange_same_length_array(sqldict)

Working with MC CCAF 1
Working with MC CCAF 2


In [6]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)
df=pd.DataFrame(sqldict)
df.to_excel(filename, index=False)
driver.quit()
sleep(3)